In [10]:
# Импорты
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier
)
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score
)

# CatBoost
from catboost import CatBoostClassifier

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

print("✓ Все библиотеки импортированы")
print(f"TensorFlow: {tf.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

✓ Все библиотеки импортированы
TensorFlow: 2.21.0
Pandas: 3.0.1
NumPy: 2.4.3


In [12]:
# Загрузка данных
df = pd.read_csv('data/students_clean.csv')

print(f"Размер датасета: {df.shape}")
print(f"\nРаспределение целевой переменной (target_encoded):")
print(df['target_encoded'].value_counts().sort_index())
print("\nРасшифровка:")
print("  0 = Dropout (Отчислен)")
print("  1 = Enrolled (Учится)")
print("  2 = Graduate (Выпускник)")

Размер датасета: (4424, 40)

Распределение целевой переменной (target_encoded):
target_encoded
0    1421
1     794
2    2209
Name: count, dtype: int64

Расшифровка:
  0 = Dropout (Отчислен)
  1 = Enrolled (Учится)
  2 = Graduate (Выпускник)


In [13]:
# Подготовка признаков и целевой переменной
X = df.drop(columns=['target_encoded'])
y = df['target_encoded']

print(f"Признаки (X): {X.shape}")
print(f"Целевая переменная (y): {y.shape}")

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Сохраняем пропорции классов
)

print(f"\nОбучающая выборка: {X_train.shape[0]} записей")
print(f"Тестовая выборка: {X_test.shape[0]} записей")

print(f"\nПропорции классов в train:")
print(y_train.value_counts(normalize=True).sort_index().apply(lambda x: f"{x:.1%}"))

Признаки (X): (4424, 39)
Целевая переменная (y): (4424,)

Обучающая выборка: 3539 записей
Тестовая выборка: 885 записей

Пропорции классов в train:
target_encoded
0    32.1%
1    17.9%
2    49.9%
Name: proportion, dtype: str


In [15]:
# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Масштабирование выполнено")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"Среднее (первые 5 признаков): {X_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Стд.откл (первые 5 признаков): {X_train_scaled.std(axis=0)[:5].round(4)}")

# Исправляем пути: всё относительно корня проекта
os.makedirs('models', exist_ok=True)

# Пересохраняем scaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Пересохраняем названия признаков
with open('models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

# Проверка
for f in os.listdir('models'):
    path = os.path.join('models', f)
    size = os.path.getsize(path)
    print(f"  {f} — {size} байт")

print("\n✓ Файлы сохранены в models/")

Масштабирование выполнено
X_train_scaled shape: (3539, 39)
Среднее (первые 5 признаков): [-0.  0. -0. -0. -0.]
Стд.откл (первые 5 признаков): [1. 1. 1. 1. 1.]
  scaler.pkl — 2408 байт
  feature_names.pkl — 976 байт

✓ Файлы сохранены в models/


In [16]:
# Функция оценки модели
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    
    # Для нейросети predict возвращает вероятности
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)
    
    f1 = f1_score(y_test, y_pred, average='macro')
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\n{'='*50}")
    print(f"МОДЕЛЬ: {model_name}")
    print(f"{'='*50}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"Accuracy:         {accuracy:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=['Dropout', 'Enrolled', 'Graduate']))
    return f1, accuracy

# Словарь для результатов
results = {}

print("✓ Функция evaluate_model готова")

✓ Функция evaluate_model готова


In [18]:
# ML1: Logistic Regression (классическая модель)
lr_model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    C=1.0,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

f1, acc = evaluate_model(lr_model, X_test_scaled, y_test, "Logistic Regression")
results['Logistic Regression'] = {'F1': f1, 'Accuracy': acc}

# Сохранение
with open('models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print("✓ Сохранена: models/logistic_regression.pkl")


МОДЕЛЬ: Logistic Regression
F1-score (macro): 0.6797
Accuracy:         0.7684

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.79      0.77      0.78       284
    Enrolled       0.52      0.32      0.40       159
    Graduate       0.81      0.93      0.86       442

    accuracy                           0.77       885
   macro avg       0.70      0.67      0.68       885
weighted avg       0.75      0.77      0.75       885

✓ Сохранена: models/logistic_regression.pkl


In [19]:
# ML2: Gradient Boosting (бустинг)
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)

f1, acc = evaluate_model(gb_model, X_test_scaled, y_test, "Gradient Boosting")
results['Gradient Boosting'] = {'F1': f1, 'Accuracy': acc}

# Сохранение
with open('models/gradient_boosting.pkl', 'wb') as f:
    pickle.dump(gb_model, f)
print("✓ Сохранена: models/gradient_boosting.pkl")


МОДЕЛЬ: Gradient Boosting
F1-score (macro): 0.7158
Accuracy:         0.7751

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.81      0.75      0.78       284
    Enrolled       0.55      0.48      0.51       159
    Graduate       0.82      0.90      0.86       442

    accuracy                           0.78       885
   macro avg       0.73      0.71      0.72       885
weighted avg       0.77      0.78      0.77       885

✓ Сохранена: models/gradient_boosting.pkl


In [ ]:
# ML3: CatBoost (продвинутый бустинг)
# CatBoost НЕ требует масштабирования — подаём исходные данные
cb_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    random_seed=42,
    verbose=100
)

cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

f1, acc = evaluate_model(cb_model, X_test, y_test, "CatBoost")
results['CatBoost'] = {'F1': f1, 'Accuracy': acc}

# Сохранение (встроенный формат CatBoost)
cb_model.save_model('models/catboost_model.cbm')
print("✓ Сохранена: models/catboost_model.cbm")

0:	learn: 1.0196307	test: 1.0190949	best: 1.0190949 (0)	total: 56.4ms	remaining: 28.1s
100:	learn: 0.4558081	test: 0.5529591	best: 0.5529591 (100)	total: 761ms	remaining: 3.01s
200:	learn: 0.3718050	test: 0.5479080	best: 0.5465990 (165)	total: 1.74s	remaining: 2.59s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5465989625
bestIteration = 165

Shrink model to first 166 iterations.

МОДЕЛЬ: CatBoost
F1-score (macro): 0.1620
Accuracy:         0.3209

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.32      1.00      0.49       284
    Enrolled       0.00      0.00      0.00       159
    Graduate       0.00      0.00      0.00       442

    accuracy                           0.32       885
   macro avg       0.11      0.33      0.16       885
weighted avg       0.10      0.32      0.16       885

✓ Сохранена: models/catboost_model.cbm


In [21]:
# Исправленная функция оценки
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    
    # Приводим к 1D массиву
    y_pred = np.array(y_pred).flatten()
    
    # Если значения — вероятности (нейросеть), берём argmax
    if y_pred.dtype == np.float32 or y_pred.dtype == np.float64:
        if y_pred.max() <= 1.0 and len(set(y_pred)) > 10:
            # Это значит, что predict вернул не классы — пересчитаем
            pass
    
    y_pred = y_pred.astype(int)
    
    f1 = f1_score(y_test, y_pred, average='macro')
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\n{'='*50}")
    print(f"МОДЕЛЬ: {model_name}")
    print(f"{'='*50}")
    print(f"F1-score (macro): {f1:.4f}")
    print(f"Accuracy:         {accuracy:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=['Dropout', 'Enrolled', 'Graduate']))
    return f1, accuracy

# Переоцениваем CatBoost (модель уже обучена)
f1, acc = evaluate_model(cb_model, X_test, y_test, "CatBoost")
results['CatBoost'] = {'F1': f1, 'Accuracy': acc}


МОДЕЛЬ: CatBoost
F1-score (macro): 0.7037
Accuracy:         0.7774

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.81      0.77      0.79       284
    Enrolled       0.55      0.40      0.46       159
    Graduate       0.81      0.92      0.86       442

    accuracy                           0.78       885
   macro avg       0.72      0.69      0.70       885
weighted avg       0.76      0.78      0.77       885



In [22]:
# ML4: Random Forest (бэггинг)
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

f1, acc = evaluate_model(rf_model, X_test_scaled, y_test, "Random Forest")
results['Random Forest'] = {'F1': f1, 'Accuracy': acc}

# Сохранение
with open('models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("✓ Сохранена: models/random_forest.pkl")


МОДЕЛЬ: Random Forest
F1-score (macro): 0.7117
Accuracy:         0.7887

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.81      0.77      0.79       284
    Enrolled       0.63      0.38      0.47       159
    Graduate       0.81      0.95      0.87       442

    accuracy                           0.79       885
   macro avg       0.75      0.70      0.71       885
weighted avg       0.78      0.79      0.77       885

✓ Сохранена: models/random_forest.pkl


In [23]:
# ML5: Stacking Classifier
# Это может занять пару минут из-за cross-validation
print("Обучение Stacking (может занять 1-2 минуты)...")

base_estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

stacking_model.fit(X_train_scaled, y_train)

f1, acc = evaluate_model(stacking_model, X_test_scaled, y_test, "Stacking")
results['Stacking'] = {'F1': f1, 'Accuracy': acc}

# Сохранение
with open('models/stacking_model.pkl', 'wb') as f:
    pickle.dump(stacking_model, f)
print("✓ Сохранена: models/stacking_model.pkl")

Обучение Stacking (может занять 1-2 минуты)...

МОДЕЛЬ: Stacking
F1-score (macro): 0.7191
Accuracy:         0.7831

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.81      0.78      0.79       284
    Enrolled       0.59      0.44      0.51       159
    Graduate       0.82      0.91      0.86       442

    accuracy                           0.78       885
   macro avg       0.74      0.71      0.72       885
weighted avg       0.77      0.78      0.77       885

✓ Сохранена: models/stacking_model.pkl


In [24]:
# ML6: Neural Network (Keras)
# Подготовка целевой переменной (one-hot encoding)
y_train_cat = to_categorical(y_train, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

# Архитектура сети
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(3, activation='softmax')
])

nn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

nn_model.summary()

E0000 00:00:1773114695.139809   27430 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,451 (64.26 KB)

 Trainable params: 16,003 (62.51 KB)

 Non-trainable params: 448 (1.75 KB)

In [25]:
# Обучение нейросети
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = nn_model.fit(
    X_train_scaled, y_train_cat,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nОбучение завершено на эпохе: {len(history.history['loss'])}")

Epoch 1/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.5606 - loss: 1.1190 - val_accuracy: 0.7316 - val_loss: 0.6926
Epoch 2/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6913 - loss: 0.7843 - val_accuracy: 0.7458 - val_loss: 0.6108
Epoch 3/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7125 - loss: 0.7342 - val_accuracy: 0.7613 - val_loss: 0.5797
Epoch 4/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7149 - loss: 0.7102 - val_accuracy: 0.7613 - val_loss: 0.5618
Epoch 5/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7428 - loss: 0.6508 - val_accuracy: 0.7613 - val_loss: 0.5623
Epoch 6/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7510 - loss: 0.6534 - val_accuracy: 0.7669 - val_loss: 0.5576
Epoch 7/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7499 - loss: 0.6332 - val_accuracy: 0.7740 - val_loss: 0.5528
Epoch 8/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7471 - loss: 0.6258 - val_accuracy: 0.7698 

In [26]:
# Оценка нейросети
# Для нейросети predict возвращает вероятности — нужен argmax
y_pred_nn = nn_model.predict(X_test_scaled)
y_pred_nn = np.argmax(y_pred_nn, axis=1)

f1 = f1_score(y_test, y_pred_nn, average='macro')
accuracy = accuracy_score(y_test, y_pred_nn)

print(f"\n{'='*50}")
print(f"МОДЕЛЬ: Neural Network")
print(f"{'='*50}")
print(f"F1-score (macro): {f1:.4f}")
print(f"Accuracy:         {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_nn,
                            target_names=['Dropout', 'Enrolled', 'Graduate']))

results['Neural Network'] = {'F1': f1, 'Accuracy': accuracy}

# Сохранение
nn_model.save('models/neural_network.keras')
print("✓ Сохранена: models/neural_network.keras")

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

МОДЕЛЬ: Neural Network
F1-score (macro): 0.6966
Accuracy:         0.7672

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.82      0.78      0.80       284
    Enrolled       0.50      0.40      0.44       159
    Graduate       0.80      0.89      0.85       442

    accuracy                           0.77       885
   macro avg       0.71      0.69      0.70       885
weighted avg       0.76      0.77      0.76       885

✓ Сохранена: models/neural_network.keras


In [28]:
# Итоговая сводка
print("="*50)
print("ИТОГОВАЯ СВОДКА РЕЗУЛЬТАТОВ")
print("="*50)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('F1', ascending=False)
results_df.index.name = 'Модель'
print(results_df.to_string(float_format='%.4f'))

# Сохраняем результаты
results_df.to_csv('models/model_results.csv')
print("\n✓ Результаты сохранены: models/model_results.csv")

# Проверяем все файлы в папке models
print("\nФайлы в models/:")
for f in sorted(os.listdir('models')):
    size = os.path.getsize(os.path.join('models', f))
    print(f"  {f} — {size:,} байт")

ИТОГОВАЯ СВОДКА РЕЗУЛЬТАТОВ
                        F1  Accuracy
Модель                              
Stacking            0.7191    0.7831
Gradient Boosting   0.7158    0.7751
Random Forest       0.7117    0.7887
CatBoost            0.7037    0.7774
Neural Network      0.6966    0.7672
Logistic Regression 0.6797    0.7684

✓ Результаты сохранены: models/model_results.csv

Файлы в models/:
  catboost_model.cbm — 375,112 байт
  feature_names.pkl — 976 байт
  gradient_boosting.pkl — 2,276,117 байт
  logistic_regression.pkl — 1,669 байт
  model_results.csv — 335 байт
  neural_network.keras — 248,612 байт
  random_forest.pkl — 13,632,110 байт
  scaler.pkl — 2,408 байт
  stacking_model.pkl — 11,514,723 байт
